# Run All

Runs the whole pipeline in order, from the raw CSV files to the deep-dive analysis:

| Step | Notebook |
| --- | --- |
| 1 | `01_data_setup.ipynb` |
| 2 | `02_data_quality.ipynb` |
| 3 | `03_data_cleaning.ipynb` |
| 4 | `04_delay_analysis.ipynb` |
| 5 | `05_deep_dive_analysis.ipynb` |

Each notebook is executed in a fresh kernel and saved in place with its outputs. The run stops at the first notebook that fails.

## 1. Setup

In [5]:
import time
from pathlib import Path

import nbformat
from nbconvert.preprocessors import CellExecutionError, ExecutePreprocessor

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
PROJECT_ROOT = NOTEBOOK_DIR.parent

PIPELINE = [
    "01_data_setup.ipynb",
    "02_data_quality.ipynb",
    "03_data_cleaning.ipynb",
    "04_delay_analysis.ipynb",
    "05_deep_dive_analysis.ipynb",
]
TIMEOUT_SEC = 1200

missing = [name for name in PIPELINE if not (NOTEBOOK_DIR / name).is_file()]
assert not missing, f"Missing notebooks: {missing}"
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/ziban/github/railpulse-railway-analytics


## 2. Run the notebooks

The working directory is the notebooks folder, the same as when each notebook is opened by hand.

In [6]:
results = []

for name in PIPELINE:
    path = NOTEBOOK_DIR / name
    nb = nbformat.read(path, as_version=4)
    executor = ExecutePreprocessor(timeout=TIMEOUT_SEC, kernel_name="python3")

    start = time.time()
    try:
        executor.preprocess(nb, {"metadata": {"path": str(NOTEBOOK_DIR)}})
        status = "OK"
    except CellExecutionError as err:
        status = "FAILED"
        error = err
    finally:
        # keep the outputs, including those of a failed run, for debugging
        nbformat.write(nb, path)

    results.append((name, status, time.time() - start))
    print(f"{status:7} {name} ({results[-1][2]:.1f}s)")
    if status == "FAILED":
        raise error

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


OK      01_data_setup.ipynb (0.7s)


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


OK      02_data_quality.ipynb (0.9s)


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


OK      03_data_cleaning.ipynb (0.7s)


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


OK      04_delay_analysis.ipynb (1.2s)


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


OK      05_deep_dive_analysis.ipynb (1.9s)


## 3. Summary

In [7]:
import pandas as pd

summary = pd.DataFrame(results, columns=["notebook", "status", "seconds"]).round({"seconds": 1})
summary

,notebook,status,seconds
0,01_data_setup.ipynb,OK,0.7
1,02_data_quality.ipynb,OK,0.9
2,03_data_cleaning.ipynb,OK,0.7
3,04_delay_analysis.ipynb,OK,1.2
4,05_deep_dive_analysis.ipynb,OK,1.9
